# ARC-AGI-3 — Step 4: Learned Path Value from levels_completed

Builds on Step 3 (state-graph BFS frontier exploration). Adds a lightweight *learned* heuristic -- not a neural net (that's deliberately out of scope per the plan; Forge v5's CBAM/PER/MCTS stack is overkill for where we are) -- using a signal Steps 2 and 3 completely ignored: **`FrameData.levels_completed`**, the actual field the competition scores on.

**The gap this closes:** Steps 2-3 only know "did this action change the frame" and "is this state new to me" -- neither has any notion of *good* vs merely *novel*. An action that changes the frame into a dead end scores the same as one that advances a level.

**Why this is honestly achievable (and the previous steps' local-simulation caveat doesn't apply here):** this needs no simulated rollouts and no assumption about discoverable game source. It only uses real, live play.

**The mechanism -- simple Monte-Carlo credit assignment:**
1. Track the sequence of `(state_hash, action_id)` pairs taken since the last    level-up (or game start) -- the *episode path*.
2. When `levels_completed` increases, every pair in that path gets +1 in a    `path_value` table, then the path resets for the next level.
3. `path_value` is a table, not a neural net -- but it persists across    **in-episode RESETs** (e.g. after a GAME_OVER puts us back at the start of    level 1). That's exactly the case where it's useful: a second attempt at an    already-seen early state can reuse what worked last time, instead of treating    it as equally-novel again.
4. When picking an action, a nonzero learned `path_value` for the current    `(state, action)` pair now outranks plain frontier-novelty or change-rate --    it's the only signal in this stack that's actually tied to winning.

In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv


In [ ]:
%%writefile /kaggle/working/my_agent.py
import hashlib
import random
import time
from typing import Any, Dict, List, Optional, Set, Tuple

from arcengine import FrameData, GameAction, GameState
from agents.agent import Agent

REGION_SIZE = 8  # bucket the 64x64 coordinate grid into 8x8 regions


class MyAgent(Agent):
    """Step 4: adds learned (state, action) -> value from levels_completed on
    top of Step 3's state-graph BFS and Step 2's change-rate tracking.

    New in this step: path_value is built with simple Monte-Carlo credit
    assignment -- when levels_completed increases, every (state, action) pair
    taken since the last level-up gets credit. This table persists across
    in-episode RESETs, so a second attempt at a previously-seen early state
    (e.g. after a GAME_OVER) can reuse what's already known to work.
    """

    MAX_ACTIONS = 80
    EPSILON = 0.15
    STUCK_EPSILON = 0.5

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        seed = int(time.time() * 1000000) + hash(self.game_id) % 1000000
        random.seed(seed)
        # Step 2: action_id -> [times_tried, times_it_changed_the_frame]
        self.action_stats: Dict[int, List[int]] = {}
        # Step 2: (region_x, region_y) -> [times_tried, times_it_changed_the_frame]
        self.region_stats: Dict[Tuple[int, int], List[int]] = {}
        # Step 3: directed state graph. node_hash -> {action_id: next_node_hash}
        self.graph: Dict[str, Dict[int, str]] = {}
        # Step 3: node_hash -> set of action_ids already tried from that node
        self.tried_from: Dict[str, Set[int]] = {}
        # Step 4: learned value per (state_hash, action_id), credited on level-up.
        # Persists across in-episode RESETs -- the whole point.
        self.path_value: Dict[Tuple[str, int], float] = {}
        # Step 4: (state_hash, action_id) pairs taken since the last level-up/reset
        self.episode_path: List[Tuple[str, int]] = []
        self.prev_levels_completed: int = 0

        self.last_hash: Optional[str] = None
        self.pending_id: Optional[int] = None
        self.pending_region: Optional[Tuple[int, int]] = None

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        return latest_frame.state is GameState.WIN

    def _frame_hash(self, latest_frame: FrameData) -> str:
        return hashlib.md5(str(latest_frame.frame).encode()).hexdigest()

    def _record_outcome(self, current_hash: str, latest_frame: FrameData) -> None:
        """Attribute the frame-change outcome, the graph edge, AND any
        level-up credit to whatever we did last turn."""
        if self.pending_id is None or self.last_hash is None:
            return
        changed = current_hash != self.last_hash

        # Step 2 bookkeeping
        stats = self.action_stats.setdefault(self.pending_id, [0, 0])
        stats[0] += 1
        stats[1] += int(changed)
        if self.pending_region is not None:
            rstats = self.region_stats.setdefault(self.pending_region, [0, 0])
            rstats[0] += 1
            rstats[1] += int(changed)

        # Step 3 bookkeeping
        self.graph.setdefault(self.last_hash, {})[self.pending_id] = current_hash
        self.tried_from.setdefault(self.last_hash, set()).add(self.pending_id)

        # Step 4 bookkeeping: record this step in the current episode path
        self.episode_path.append((self.last_hash, self.pending_id))

        current_levels = getattr(latest_frame, "levels_completed", 0) or 0
        if current_levels > self.prev_levels_completed:
            # Level advanced: credit every (state, action) pair that led here.
            for pair in self.episode_path:
                self.path_value[pair] = self.path_value.get(pair, 0.0) + 1.0
            self.episode_path = []
        # A decrease (e.g. counter resets with the game) is just a counter
        # reset, not a loss of progress -- don't penalize, just resync.
        self.prev_levels_completed = current_levels

    def _score(self, stats_dict: dict, key: Any) -> float:
        tried, changed = stats_dict.get(key, [0, 0])
        if tried == 0:
            return 1.0
        return changed / tried

    def _pick_by_score(self, stats_dict: dict, keys: list, epsilon: float) -> Any:
        if random.random() < epsilon:
            return random.choice(keys)
        best_score = max(self._score(stats_dict, k) for k in keys)
        best_keys = [k for k in keys if self._score(stats_dict, k) == best_score]
        return random.choice(best_keys)

    def _pick_action(self, node_hash: str, legal_ids: list) -> int:
        # Step 4 takes priority: if any legal action from this exact state has
        # a proven positive learned value, take the best one outright -- this
        # is the only signal tied to actually winning, so it overrides mere
        # novelty or change-rate.
        valued = [(a, self.path_value[(node_hash, a)]) for a in legal_ids
                  if self.path_value.get((node_hash, a), 0.0) > 0]
        if valued:
            best_val = max(v for _, v in valued)
            best_actions = [a for a, v in valued if v == best_val]
            return random.choice(best_actions)

        # Step 3: BFS frontier -- prefer untried-from-this-node actions
        tried_here = self.tried_from.get(node_hash, set())
        untried = [a for a in legal_ids if a not in tried_here]
        if untried:
            return random.choice(untried)

        # Step 2 fallback: global change-rate, more randomness since we're stuck
        return self._pick_by_score(self.action_stats, legal_ids, self.STUCK_EPSILON)

    def choose_action(
        self, frames: list[FrameData], latest_frame: FrameData
    ) -> GameAction:
        if latest_frame.state in [GameState.NOT_PLAYED, GameState.GAME_OVER]:
            self.last_hash = None
            self.pending_id = None
            self.pending_region = None
            self.episode_path = []
            return GameAction.RESET

        current_hash = self._frame_hash(latest_frame)
        self._record_outcome(current_hash, latest_frame)

        legal_ids = [a for a in latest_frame.available_actions if a != GameAction.RESET.value]
        if not legal_ids:
            legal_ids = [a.value for a in GameAction if a is not GameAction.RESET]

        chosen_id = self._pick_action(current_hash, legal_ids)
        action = GameAction.from_id(chosen_id)
        self.pending_id = chosen_id

        if action.is_simple():
            self.pending_region = None
            tried, changed = self.action_stats.get(chosen_id, [0, 0])
            learned = self.path_value.get((current_hash, chosen_id), 0.0)
            action.reasoning = (
                f"id={chosen_id} change_rate={changed}/{tried} learned_value={learned}"
            )
        elif action.is_complex():
            n_regions = 64 // REGION_SIZE
            region_keys = [(rx, ry) for rx in range(n_regions) for ry in range(n_regions)]
            region = self._pick_by_score(self.region_stats, region_keys, self.EPSILON)
            self.pending_region = region
            x = region[0] * REGION_SIZE + random.randint(0, REGION_SIZE - 1)
            y = region[1] * REGION_SIZE + random.randint(0, REGION_SIZE - 1)
            action.set_data({"x": x, "y": y})
            tried, changed = self.region_stats.get(region, [0, 0])
            learned = self.path_value.get((current_hash, chosen_id), 0.0)
            action.reasoning = {
                "region": str(region),
                "change_rate": f"{changed}/{tried}",
                "learned_value": str(learned),
            }

        self.last_hash = current_hash
        return action


In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    !cp /kaggle/working/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type, cast
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    \"random\": Random,
    \"myagent\": MyAgent,
}
""")

    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
RECORDINGS_DIR=/kaggle/working/server_recording
""")

    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg \
        python main.py --agent myagent


In [ ]:
import os
import pandas as pd

if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
